# 03d – XGBoost Regressor
**Owner:** Member 3 (Classical Machine Learning Engineer)

Tunes an `XGBRegressor` inside the shared preprocessing `Pipeline` using `Optuna` (5-fold CV on training data only), records search/training times, saves predictions and feature importances.

In [1]:
import sys
import time
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import optuna
import pandas as pd
from sklearn.model_selection import KFold, cross_val_score
from sklearn.pipeline import Pipeline
from xgboost import XGBRegressor

optuna.logging.set_verbosity(optuna.logging.WARNING)

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import CV_FOLDS, FIGURES_DIR, MODELS_DIR, RANDOM_SEED, RESULTS_DIR
from src.evaluate import (
    compute_metrics,
    cross_validate_model,
    plot_parity,
    plot_residuals,
    save_model_run_metadata,
    save_predictions,
)
from src.features import SELECTED_FEATURES, build_preprocessor, get_xy

MODELS_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

X_train, y_train, id_train = get_xy("train")
X_val, y_val, id_val = get_xy("val")
X_test, y_test, id_test = get_xy("test")

C:\Users\nourh\.gemini\antigravity\scratch\mof-co2-uptake-prediction\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
cv_split = KFold(n_splits=CV_FOLDS, shuffle=True, random_state=RANDOM_SEED)

def objective(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 150, 550, step=100),
        "max_depth": trial.suggest_int("max_depth", 3, 7),
        "learning_rate": trial.suggest_float("learning_rate", 0.02, 0.2, log=True),
        "subsample": trial.suggest_float("subsample", 0.65, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-4, 5.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-3, 10.0, log=True),
    }
    pipe = Pipeline([
        ("preprocessor", build_preprocessor()),
        ("model", XGBRegressor(
            random_state=RANDOM_SEED,
            n_jobs=-1,
            tree_method="hist",
            **params,
        )),
    ])
    scores = cross_val_score(
        pipe, X_train, y_train, cv=cv_split, scoring="neg_root_mean_squared_error", n_jobs=1
    )
    return float(-np.mean(scores))

sampler = optuna.samplers.TPESampler(seed=RANDOM_SEED)
study = optuna.create_study(direction="minimize", sampler=sampler)

t_search_start = time.perf_counter()
study.optimize(objective, n_trials=15)
search_time = time.perf_counter() - t_search_start

print(f"Optuna search (15 trials) completed in {search_time:.2f} s")
print(f"Best CV RMSE: {study.best_value:.4f}")
print("Best Params :", study.best_params)

Optuna search (15 trials) completed in 287.80 s
Best CV RMSE: 0.9093
Best Params : {'n_estimators': 450, 'max_depth': 5, 'learning_rate': 0.07124613906967675, 'subsample': 0.8987102131109223, 'colsample_bytree': 0.9863201554924214, 'reg_alpha': 0.8195707470465986, 'reg_lambda': 0.056166760102145274}


In [3]:
best_xgb_pipe = Pipeline([
    ("preprocessor", build_preprocessor()),
    ("model", XGBRegressor(
        random_state=RANDOM_SEED,
        n_jobs=-1,
        tree_method="hist",
        **study.best_params,
    )),
])

cv_summary = cross_validate_model(best_xgb_pipe, X_train, y_train, cv=CV_FOLDS)
print("Final XGBoost 5-Fold CV:", {k: round(v, 4) for k, v in cv_summary.items()})

t_fit_start = time.perf_counter()
best_xgb_pipe.fit(X_train, y_train)
fit_time = time.perf_counter() - t_fit_start

val_pred = best_xgb_pipe.predict(X_val)
val_metrics = compute_metrics(y_val, val_pred)
print(f"Validation Metrics: {val_metrics}")

test_pred = best_xgb_pipe.predict(X_test)
test_metrics = compute_metrics(y_test, test_pred)
print(f"Test Metrics      : {test_metrics}")

joblib.dump(best_xgb_pipe, MODELS_DIR / "xgboost_best.joblib")
save_predictions(id_val, y_val, val_pred, "xgboost", "val")
save_predictions(id_test, y_test, test_pred, "xgboost", "test")
save_model_run_metadata(
    model_name="xgboost",
    display_name="XGBoost",
    training_time_sec=fit_time,
    search_time_sec=search_time,
    cv_summary=cv_summary,
    extra={"best_params": study.best_params},
)

# Extract and save feature importances
importances = best_xgb_pipe.named_steps["model"].feature_importances_
fi_df = pd.DataFrame({"feature": SELECTED_FEATURES, "importance": importances})
fi_df = fi_df.sort_values("importance", ascending=False).reset_index(drop=True)
fi_df.to_csv(RESULTS_DIR / "xgb_feature_importance.csv", index=False)

fig, ax = plt.subplots(figsize=(8.5, 6))
top20 = fi_df.head(20).iloc[::-1]
ax.barh(top20["feature"], top20["importance"], color="#e76f51")
ax.set_title("XGBoost Top 20 Feature Importances (Gain)")
ax.set_xlabel("Feature Importance")
fig.tight_layout()
fig.savefig(FIGURES_DIR / "xgb_feature_importance.png", dpi=200)
plt.close(fig)

plot_parity(y_test, test_pred, "xgboost", split="test")
plot_residuals(y_test, test_pred, "xgboost", split="test")
print(fi_df.head(10))

Final XGBoost 5-Fold CV: {'r2_mean': 0.6449, 'r2_std': 0.0331, 'rmse_mean': 0.9112, 'rmse_std': 0.0466, 'mae_mean': 0.6092, 'mae_std': 0.0288}


Validation Metrics: {'r2': 0.7027092825992174, 'rmse': 0.9152120279710849, 'mae': 0.6130149034874319}
Test Metrics      : {'r2': 0.6510056955166381, 'rmse': 0.8643385066532955, 'mae': 0.5963964094988796}


              feature  importance
0      mc_CRY-I-1-all    0.050813
1    mc_CRY-chi-2-all    0.046776
2    mc_CRY-chi-0-all    0.034014
3      sum-lc-S-0-all    0.025163
4      mc_CRY-T-0-all    0.023119
5           f-lig-Z-1    0.019776
6  sum-lc-alpha-0-all    0.017850
7      PONAV [cm^3/g]    0.014932
8      mc_CRY-S-1-all    0.014850
9      mc_CRY-Z-0-all    0.013904
